# ERA5 Single-Level Monthly Means — South Asia (Objective 2)

Second ERA5 single-level download, cropped to **South Asia (5°N–40°N, 60°E–100°E)**
on a **2°×2°** grid. The first ERA5 single-level pull covered the tropical Pacific;
this one re-uses the same variables (plus 2 m temperature) over the monsoon domain,
saving a CSV + JSON metadata and uploading to a dedicated Google Drive folder.

| Field | ERA5 variable | Units |
|-------|---------------|-------|
| `sst`   | `sea_surface_temperature` | K |
| `t2m`   | `2m_temperature` | K |
| `msl`   | `mean_sea_level_pressure` | Pa |
| `metss` | `mean_surface_eastward_turbulent_stress` (τx) | N m⁻² |
| `mntss` | `mean_surface_northward_turbulent_stress` (τy) | N m⁻² |
| `ttr`   | `top_net_thermal_radiation` (OLR) | J m⁻² |

**Dataset:** `reanalysis-era5-single-levels-monthly-means` ·
[CDS page](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-monthly-means)

## 1. Imports & configuration

In [4]:
import os
import json
import time
import zipfile
import glob

import numpy as np
import pandas as pd
import xarray as xr
import cdsapi

# --- Local output folder (separate from the tropical-Pacific ERA5 data) ---
DATA_DIR = "../../data/ERA5_single_levels_southasia"
os.makedirs(DATA_DIR, exist_ok=True)

# --- CDS dataset ---
DATASET = "reanalysis-era5-single-levels-monthly-means"

# --- Region: South Asia (5N-40N, 60E-100E) as area = [North, West, South, East] ---
AREA = [40, 60, 5, 100]
GRID = [2.0, 2.0]

# --- Time range (monthly means) ---
START_YEAR = 1980
END_YEAR = 2025

# --- Same single-level variables as the first ERA5 pull, plus 2 m temperature ---
# NOTE: the turbulent-stress fields use the canonical ERA5 names
# "mean_eastward/northward_turbulent_surface_stress" (the reordered spelling
# "mean_surface_eastward_turbulent_stress" is rejected by MARS as ambiguous).
VARIABLES = [
    "sea_surface_temperature",                    # SST
    "2m_temperature",                             # T2M
    "mean_sea_level_pressure",                    # MSLP
    "mean_eastward_turbulent_surface_stress",     # tau_x
    "mean_northward_turbulent_surface_stress",    # tau_y
    "top_net_thermal_radiation",                  # OLR
]

print(f"Output dir : {DATA_DIR}")
print(f"Dataset    : {DATASET}")
print(f"Region     : N{AREA[0]} W{AREA[1]} S{AREA[2]} E{AREA[3]}  grid {GRID}")
print(f"Time range : {START_YEAR}-{END_YEAR} monthly")
print(f"Variables  : {len(VARIABLES)}")

Output dir : ../../data/ERA5_single_levels_southasia
Dataset    : reanalysis-era5-single-levels-monthly-means
Region     : N40 W60 S5 E100  grid [2.0, 2.0]
Time range : 1980-2025 monthly
Variables  : 6


## 2. Download from CDS\n\nAll variables in a single monthly-means request. An existing valid file is reused,\nand the retrieve is retried up to 3 times on transient network errors.

In [5]:
def download_variable(client, variable, retries=3, wait=60):
    """Download one ERA5 single-level variable as its own NetCDF; reuse if present.

    One request per variable keeps each retrieve well under the CDS cost limit
    (a single 6-variable x 552-month request is rejected as "too large").
    """
    slug = variable.replace("/", "_")
    path = os.path.join(DATA_DIR, f"era5_sl_{slug}_{START_YEAR}_{END_YEAR}.nc")
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print(f"  {variable}: already downloaded")
        return path

    request = {
        "product_type": ["monthly_averaged_reanalysis"],
        "variable": [variable],
        "year": [str(y) for y in range(START_YEAR, END_YEAR + 1)],
        "month": [f"{m:02d}" for m in range(1, 13)],
        "time": ["00:00"],
        "area": AREA,           # [N, W, S, E]
        "grid": GRID,           # 2x2 degrees
        "data_format": "netcdf",
        "download_format": "unarchived",
    }

    for attempt in range(1, retries + 1):
        try:
            client.retrieve(DATASET, request, path)
            print(f"  {variable}: downloaded")
            return path
        except Exception as e:
            if attempt == retries:
                raise
            print(f"  {variable}: retrieve failed (attempt {attempt}/{retries}): {e}")
            print(f"  retrying in {wait}s ...")
            time.sleep(wait)


client = cdsapi.Client()

paths = {}
for variable in VARIABLES:
    paths[variable] = download_variable(client, variable)

print("\nAll variables downloaded.")

  sea_surface_temperature: already downloaded
  2m_temperature: already downloaded
  mean_sea_level_pressure: already downloaded


2026-06-27 16:04:48,466 INFO Request ID is 1d0b4dfe-d5c5-4afc-abb0-dc6540c91fc3
2026-06-27 16:04:48,714 INFO status has been updated to accepted
2026-06-27 16:05:32,406 INFO status has been updated to running
2026-06-27 16:06:20,533 INFO status has been updated to successful


  mean_eastward_turbulent_surface_stress: downloaded


2026-06-27 16:06:34,002 INFO Request ID is ad70ea0c-1f18-412a-a694-8b8dd4ebf8c6
2026-06-27 16:06:34,304 INFO status has been updated to accepted
2026-06-27 16:06:47,070 INFO status has been updated to running
2026-06-27 16:07:55,109 INFO status has been updated to successful
Recovering from connection error [HTTPSConnectionPool(host='cds.climate.copernicus.eu', port=443): Read timed out. (read timeout=60)], attempt 1 of 500
Retrying in 120 seconds
Recovering from connection error [HTTPSConnectionPool(host='cds.climate.copernicus.eu', port=443): Read timed out. (read timeout=60)], attempt 2 of 500
Retrying in 120 seconds
Recovering from connection error [HTTPSConnectionPool(host='cds.climate.copernicus.eu', port=443): Read timed out. (read timeout=60)], attempt 3 of 500
Retrying in 120 seconds


KeyboardInterrupt: 

## 3. Open, save CSV + metadata\n\nThe download may arrive as a single NetCDF or (occasionally) a zip of NetCDFs;\nboth are handled. The dataset is flattened to a long-format CSV plus a metadata JSON.

In [6]:
def open_era5(path):
    """Open an ERA5 download that may be a single NetCDF or a zip of NetCDFs."""
    if zipfile.is_zipfile(path):
        extract_dir = path + "_extracted"
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(path) as z:
            z.extractall(extract_dir)
        nc_files = sorted(glob.glob(os.path.join(extract_dir, "*.nc")))
        ds = xr.open_mfdataset(nc_files, combine="by_coords")
    else:
        ds = xr.open_dataset(path)

    # Normalise the time coordinate name (newer CDS files use 'valid_time')
    if "valid_time" in ds.coords and "time" not in ds.coords:
        ds = ds.rename({"valid_time": "time"})
    # Drop bookkeeping coords if present
    for c in ("number", "expver"):
        if c in ds.coords:
            ds = ds.drop_vars(c)
    return ds


# Open every per-variable file and merge into one dataset on (time, lat, lon)
datasets = [open_era5(paths[v]) for v in VARIABLES]
ds = xr.merge(datasets, compat="override", join="outer").sortby("time")

# Long-format DataFrame: time, latitude, longitude lead the geophysical columns
df = ds.to_dataframe().reset_index()
lead = [c for c in ("time", "latitude", "longitude") if c in df.columns]
rest = [c for c in df.columns if c not in lead]
df = df[lead + rest]

csv_path = os.path.join(DATA_DIR, f"era5_single_levels_southasia_{START_YEAR}_{END_YEAR}.csv")
df.to_csv(csv_path, index=False)
print(f"CSV saved: {csv_path} ({len(df)} rows)")

# --- Metadata JSON ---
times = pd.to_datetime(ds.time.values)
metadata = {
    "dataset": DATASET,
    "region": {"name": "South Asia", "north": AREA[0], "west": AREA[1], "south": AREA[2], "east": AREA[3]},
    "grid": {"resolution_deg": GRID,
              "n_lat": int(ds.sizes["latitude"]),
              "n_lon": int(ds.sizes["longitude"])},
    "latitude_range": [float(ds.latitude.min()), float(ds.latitude.max())],
    "longitude_range": [float(ds.longitude.min()), float(ds.longitude.max())],
    "time_range": [str(times[0]), str(times[-1])],
    "n_time": int(ds.sizes["time"]),
    "dimensions": {k: int(v) for k, v in ds.sizes.items()},
    "variables": {
        v: {
            "dims": list(ds[v].dims),
            "shape": list(ds[v].shape),
            "units": ds[v].attrs.get("units", ""),
            "long_name": ds[v].attrs.get("long_name", ds[v].attrs.get("standard_name", "")),
        }
        for v in ds.data_vars
    },
    "csv_rows": int(len(df)),
}
meta_path = os.path.join(DATA_DIR, f"era5_single_levels_southasia_{START_YEAR}_{END_YEAR}_metadata.json")
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2, default=str)
print(f"Metadata saved: {meta_path}")

KeyError: 'mean_northward_turbulent_surface_stress'

## 4. Upload to Google Drive (dedicated folder)\n\nUpload only the two derived files (CSV + metadata) into a new\n`ERA5_single_levels_southasia` folder under the shared Drive parent.

In [ ]:
import pickle
import mimetypes
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

SCOPES = ["https://www.googleapis.com/auth/drive"]
TOKEN_PATH = "../../token.pickle"
CLIENT_SECRETS = "../../client_secrets.json"
DRIVE_PARENT_ID = "1zLJgkYkrM1LRgtl7v5sfAQ4aits9zfUq"

# Dedicated Drive folder for this dataset
DRIVE_FOLDER_NAME = "ERA5_single_levels_southasia"

FILES_TO_UPLOAD = [csv_path, meta_path]


def get_drive_service():
    creds = None
    if os.path.exists(TOKEN_PATH):
        with open(TOKEN_PATH, "rb") as f:
            creds = pickle.load(f)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRETS, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(TOKEN_PATH, "wb") as f:
            pickle.dump(creds, f)
    return build("drive", "v3", credentials=creds)


def get_or_create_folder(service, name, parent_id):
    query = (
        f"name = '{name}' and mimeType = 'application/vnd.google-apps.folder' "
        f"and '{parent_id}' in parents and trashed = false"
    )
    resp = service.files().list(q=query, spaces="drive", fields="files(id, name)").execute()
    files = resp.get("files", [])
    if files:
        return files[0]["id"]
    folder = service.files().create(
        body={"name": name, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]},
        fields="id",
    ).execute()
    return folder["id"]


def upload_file(service, path, folder_id):
    name = os.path.basename(path)
    mimetype = mimetypes.guess_type(path)[0] or "application/octet-stream"
    media = MediaFileUpload(path, mimetype=mimetype, resumable=True)
    request = service.files().create(
        body={"name": name, "parents": [folder_id]},
        media_body=media,
        fields="id, name",
    )
    response = None
    while response is None:
        _, response = request.next_chunk()
    return response["id"]


service = get_drive_service()
folder_id = get_or_create_folder(service, DRIVE_FOLDER_NAME, DRIVE_PARENT_ID)
print(f"Drive folder: {DRIVE_FOLDER_NAME} (id: {folder_id})")

uploaded = {}
for path in FILES_TO_UPLOAD:
    fid = upload_file(service, path, folder_id)
    uploaded[os.path.basename(path)] = fid
    print(f"  Uploaded: {os.path.basename(path)} (id: {fid})")

print("Done.")

## 5. Validation

In [ ]:
lat = np.sort(df["latitude"].unique())
lon = np.sort(df["longitude"].unique())
n_time = df["time"].nunique()
expected_rows = n_time * lat.size * lon.size
field_cols = [c for c in df.columns if c not in ("time", "latitude", "longitude")]

print("--- Validation ---")
print(f"Columns      : {list(df.columns)}")
print(f"Latitudes    : {lat.min()}..{lat.max()} step {np.unique(np.diff(lat))} ({lat.size} pts)")
print(f"Longitudes   : {lon.min()}..{lon.max()} step {np.unique(np.diff(lon))} ({lon.size} pts)")
print(f"Time steps   : {n_time} ({df['time'].min()} .. {df['time'].max()})")
print(f"CSV rows     : {len(df)} (expected {n_time} x {lat.size} x {lon.size} = {expected_rows})")

print("\nField ranges (NaNs ignored; SST is NaN over land):")
for c in field_cols:
    print(f"  {c:7s} min/max: {df[c].min():.4g} .. {df[c].max():.4g}")

print(f"\nDrive folder '{DRIVE_FOLDER_NAME}' uploads: {list(uploaded.keys())}")